# MLflow Example: Tracking Two Model Training Runs

The teacher's notebook, fixed so it runs, and extended with the parts `15-MLFlow.pdf` describes but the original notebook lacked: registering the models, loading one back from the registry, and serving it. Cells marked **(added)** are new.

In [ ]:
%pip install scikit-learn

In [ ]:
# === Model Training and Experimentation ===

# Import necessary libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits


In [ ]:

# Load dataset
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

# Define a function to train a model and return its accuracy
def train_model(n_estimators, max_depth):
    # Train a RandomForestClassifier
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    print(f"Model with n_estimators={n_estimators}, max_depth={max_depth} achieved accuracy={accuracy:.4f}")
    return model, accuracy

In [ ]:

# Train models with different hyperparameter configurations
print("Training Model 1...")
model_1, acc_1 = train_model(n_estimators=20, max_depth=5)

In [ ]:
print("\nTraining Model 2...")
model_2, acc_2 = train_model(n_estimators=100, max_depth=10)


Now adding MLFlow

In [ ]:
%pip install mlflow

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits

In [ ]:
%pip install dagshub

In [ ]:
# get this code from dagshub (Remote > Experiments)
import os
import dagshub

DAGSHUB_OWNER = "HadiHz88"
DAGSHUB_REPO = "aide505-mlflow-dagshub"

# MLflow also honours MLFLOW_TRACKING_URI: set it to try the notebook against a local
# `mlflow server` instead of DagsHub.
if "MLFLOW_TRACKING_URI" not in os.environ:
    dagshub.init(repo_owner=DAGSHUB_OWNER, repo_name=DAGSHUB_REPO, mlflow=True)
print("Tracking URI:", mlflow.get_tracking_uri())

In [ ]:
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

In [ ]:
# Set MLflow experiment name. This will get created if it doesn't exist
experiment_name = "RandomForest_V1.0.1"
mlflow.set_experiment(experiment_name)

# Every logged model is also registered under this name, as a new version (added)
REGISTERED_MODEL = "digits-random-forest"


In [ ]:
# Define a function to train a model, log parameters and metrics to MLflow
def train_and_log_model(n_estimators, max_depth, run_name):
    with mlflow.start_run(run_name=run_name):  # run_name names the run in the UI
        # Train a RandomForestClassifier
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Make predictions
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)

        # Log parameters, metrics, and the model itself
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.sklearn.log_model(
            model,
            name="model",                          # model names cannot contain dots
            serialization_format="cloudpickle",    # the default (skops) rejects random forests
            input_example=X_test[:1],              # records the input shape: 64 pixel values
            registered_model_name=REGISTERED_MODEL,  # adds a version in the model registry
        )

        print(f"Logged RandomForest model with n_estimators={n_estimators}, max_depth={max_depth}, accuracy={accuracy:.4f}")


In [ ]:
# Train and log Model 1
print("Training and Logging Model 1...")
train_and_log_model(n_estimators=20, max_depth=5, run_name="random_forest_model_run_1")

In [ ]:
# Train and log Model 2
print("\nTraining and Logging Model 2...")
train_and_log_model(n_estimators=100, max_depth=10, run_name="random_forest_model_run_V1.0.2")

## Model registry (added)

Each `log_model` call above also registered the model, so `digits-random-forest` now has one version per training run. The **latest** version is the one with the highest number, i.e. the model registered last. That is the one the Docker image downloads.

In [ ]:
client = MlflowClient()
versions = client.search_model_versions(f"name='{REGISTERED_MODEL}'")
for v in sorted(versions, key=lambda v: int(v.version)):
    run = client.get_run(v.run_id)
    print(f"version {v.version}: run {run.info.run_name}, accuracy {run.data.metrics['accuracy']:.4f}")

latest = max(versions, key=lambda v: int(v.version))
print("latest version:", latest.version)

## Load a model from the registry (added)

`models:/<name>/<version>` points at a registered version. MLflow downloads it and rebuilds the scikit-learn object.

In [ ]:
loaded = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL}/{latest.version}")
print("predicted:", loaded.predict(X_test[:10]))
print("actual:   ", y_test[:10])

## Serve the model (added)

In a terminal where the same environment is active, and with the same tracking settings as this notebook (on DagsHub: `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME` and `MLFLOW_TRACKING_PASSWORD`):

```bash
mlflow models serve -m "models:/digits-random-forest/2" -p 5003 --env-manager local
```

`--env-manager local` uses the current environment instead of building a new one (which is what asks for `virtualenv`). Then send one image, 64 pixel values:

```bash
curl -s localhost:5003/invocations -H 'Content-Type: application/json' \
  -d '{"inputs": [[0,0,0,7,12,0,0,0,0,0,4,16,8,0,0,0,0,0,12,11,0,0,0,0,0,0,15,10,8,6,1,0,0,0,15,16,8,10,8,0,0,0,14,7,0,0,12,0,0,0,8,11,0,5,16,2,0,0,0,9,14,14,5,0]]}'
```

In [ ]:
# Instructions to visualize results
# `mlflow ui` would start a local server that knows nothing about runs stored on DagsHub.
print("\nTo view the results, open the repository on DagsHub:")
print(f"https://dagshub.com/{DAGSHUB_OWNER}/{DAGSHUB_REPO}/experiments")